# Module 1: Build Your First ML Model

In this notebook, we build our first end-to-end ML pipeline — from raw data to trained model to evaluation. The goal is to understand the fundamental workflow that every ML project follows, regardless of complexity.

**Key concepts:** supervised learning, train/test split, feature scaling, model selection, cross-validation

## 1. Setup and Imports

We start with the core ML stack: `numpy` for numerical operations, `pandas` for data manipulation, and `scikit-learn` for the ML pipeline. These three libraries form the foundation of virtually every ML project.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded successfully")

## 2. Data Loading and Exploration

We use the California Housing dataset — a real-world regression problem where we predict median house values from features like median income, house age, and location. This is a good starting dataset because:
- It has both numerical features with different scales (requiring normalization)
- Features have varying predictive power (good for understanding feature importance)
- It's large enough (20,640 samples) to be realistic but small enough to train quickly

In [ ]:
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"Target: Median house value (in $100,000s)")
print(f"\nTarget statistics:")
print(f"  Mean: ${y.mean() * 100000:,.0f}")
print(f"  Std:  ${y.std() * 100000:,.0f}")
print(f"  Range: ${y.min() * 100000:,.0f} - ${y.max() * 100000:,.0f}")

### Why explore before modeling?

Looking at feature distributions helps us understand:
1. **Scale differences** — MedInc ranges 0-15, AveRooms ranges 0-140. Without scaling, features with larger magnitudes dominate distance-based or regularized models.
2. **Potential outliers** — Extreme values in AveRooms or Population could skew our model.
3. **Feature correlations** — Highly correlated features add redundancy without information.

In [ ]:
print("Feature statistics:")
print(X.describe().round(2).to_string())
print(f"\nCorrelation with target (top features):")
correlations = X.corrwith(y).abs().sort_values(ascending=False)
for feat, corr in correlations.items():
    print(f"  {feat}: {corr:.3f}")

## 3. Train/Test Split

**Why 80/20?** This is the standard split that balances having enough training data for the model to learn patterns while keeping enough test data for a reliable evaluation. With 20,640 samples, our test set of ~4,128 is large enough for stable metrics.

**Why `random_state=42`?** Reproducibility. Anyone running this notebook gets the same split, making results comparable. The specific number doesn't matter — it's just a convention.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

## 4. Feature Scaling

**Why StandardScaler?** Linear models (and regularized variants like Ridge/Lasso) are sensitive to feature scale. StandardScaler transforms each feature to have mean=0 and std=1.

**Critical:** We `fit` the scaler on training data only, then `transform` both train and test. If we fit on the full dataset, we'd leak test set statistics into training — a subtle but common mistake called **data leakage**.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform on train
X_test_scaled = scaler.transform(X_test)         # only transform on test

print("After scaling (first feature):")
print(f"  Train mean: {X_train_scaled[:, 0].mean():.6f} (should be ~0)")
print(f"  Train std:  {X_train_scaled[:, 0].std():.6f} (should be ~1)")
print(f"  Test mean:  {X_test_scaled[:, 0].mean():.6f} (close to 0, not exact)")

## 5. Model Training and Comparison

We compare three linear models to understand the effect of regularization:

| Model | Regularization | Effect |
|-------|---------------|--------|
| **Linear Regression** | None | Fits all features equally — can overfit with many features |
| **Ridge (L2)** | Shrinks coefficients toward zero | Reduces overfitting, keeps all features |
| **Lasso (L1)** | Drives some coefficients to exactly zero | Feature selection built-in |

**Why start with linear models?** They're fast, interpretable, and establish a baseline. If a linear model does well, you might not need the complexity of tree-based or neural models.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge (alpha=1.0)": Ridge(alpha=1.0),
    "Lasso (alpha=0.01)": Lasso(alpha=0.01),
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="r2")

    results[name] = {"RMSE": rmse, "R2": r2, "CV_R2": cv_scores.mean(), "CV_std": cv_scores.std()}
    print(f"{name}:")
    print(f"  RMSE: {rmse:.4f} (${rmse * 100000:,.0f} average error)")
    print(f"  R\u00b2:   {r2:.4f}")
    print(f"  CV R\u00b2: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print()

## 6. Analysis: What Do These Results Tell Us?

**Interpreting the metrics:**
- **RMSE of ~0.73** means our predictions are off by about $73,000 on average — not great for a home price, but reasonable for a simple linear model with no feature engineering.
- **R^2 of ~0.60** means we explain about 60% of the variance in house prices. The remaining 40% comes from factors not in our features (school quality, neighborhood safety, etc.).
- **CV R^2 being close to test R^2** tells us we're not overfitting — the model generalizes consistently.

**Key insight:** Ridge and Lasso perform nearly identically to basic linear regression here. This tells us the dataset doesn't have severe multicollinearity or too many irrelevant features — regularization has little to correct. With higher-dimensional data or more features, Ridge/Lasso would show more benefit.

**Next steps to improve:** Feature engineering (interactions, polynomial features), tree-based models (Random Forest, XGBoost), or handling the geographic features (latitude/longitude) more intelligently.

## 7. Feature Importance

Linear model coefficients directly tell us which features matter most. After scaling, coefficient magnitude = feature importance.

In [ ]:
lr = models["Linear Regression"]
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lr.coef_,
    "abs_coefficient": np.abs(lr.coef_),
}).sort_values("abs_coefficient", ascending=False)

print("Feature importance (by |coefficient|):")
for _, row in feature_importance.iterrows():
    direction = "+" if row["coefficient"] > 0 else "-"
    print(f"  {direction} {row['feature']:<15} {row['abs_coefficient']:.4f}")

print(f"\nMedInc (median income) is the strongest predictor.")
print(f"Latitude has a negative coefficient - moving north in CA generally means lower prices.")

## Key Takeaways

1. **Always explore before modeling** — understanding feature scales, distributions, and correlations informs preprocessing decisions
2. **Scale features for linear models** — and always fit the scaler on train data only
3. **Cross-validate** — a single train/test split can be misleading; CV gives confidence intervals
4. **Start simple** — linear models establish a baseline and are fully interpretable
5. **Regularization helps when needed** — with clean, low-dimensional data, the benefit is minimal